# 🩺 Diabetes Prediction — Complete 4-Milestone Demo Day Project
## CTGAN edition: **768 real + 1,232 synthetic = exactly 2,000 records**

**Goal:** Predict diabetes risk (`Outcome`: 0 negative; 1 positive), prioritizing **recall > 80%** while requiring **accuracy > 60%**. This is an educational machine-learning project, **not a medical diagnostic tool**.

### Four milestones
1. **Milestone 1:** Problem definition and original dataset (768 real observations).
2. **Milestone 2:** Dataset inspection, duplicates, missing-value handling.
3. **Milestone 3:** Leak-free real-only splits, EDA, CTGAN synthesis of **1,232 new records**, quality analysis including three correlation heatmaps.
4. **Milestone 4:** Compare Logistic Regression, KNN, Decision Tree and Random Forest, both real-only and CTGAN-augmented. Select the **best CTGAN-augmented model** by first requiring **recall > 80%** and **accuracy > 60%** on real validation data, then **minimizing false positives** (maximizing specificity). F2 is a tie-breaker, not the primary selection objective. Evaluate once on untouched real test records. **After evaluation, retrain the selected model from scratch on ALL 2,000 rows**, and save that final full-data model.

### Why there are two model-training stages
- **Evaluation stage:** CTGAN learns from only real training records. Each augmented candidate fits on those training records plus the 1,232 synthetics. Real validation selects the threshold; real test evaluates its performance **before** the holdout is used for training.
- **Final full-data stage:** After reporting the honest held-out test performance, refit the **same model family on all 768 original real + all 1,232 CTGAN-generated rows = 2,000 rows**. All original records (including those formerly held out) now contribute to the final model. Accordingly, the previously reported test scores describe the *pre-refit evaluation model*, **not** an independent test of the final 2,000-row model.

**A recall of 1.00 can occur on a small holdout**, but should never be called perfect clinical performance. Inspect the false positives, specificity, accuracy, and confidence interval. The F2 metric weights recall more than precision while avoiding thresholds chosen solely to maximize recall.

Run in **Google Colab** with `diabetes_dataset.csv` uploaded. T4 GPU can speed CTGAN. No ZIP creation or automatic folder download is included.


## ⚙️ Setup — install dependencies

CTGAN is **required** here, not an optional replacement by SMOTE. On a fresh Colab runtime, execute this installation cell first. If installation fails due to a temporary network issue, retry before continuing. No fake synthetic data will be substituted.

In [ ]:
%pip -q install "ctgan>=0.12,<0.14" "scikit-learn>=1.4" pandas numpy scipy matplotlib seaborn joblib cloudpickle

## Settings and imports

To change the quantity generated or the CTGAN training duration, edit only the configuration values below. `CTGAN_EPOCHS=200` is a starting point, **not a guarantee** of high-fidelity synthetic data. If the model underfits, consider 300–500 epochs, then re-run **all** downstream evaluations. Training may take several minutes, particularly without a GPU.

In [ ]:
from pathlib import Path
from IPython.display import display
import json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import cloudpickle
import torch
from scipy.stats import ks_2samp, binomtest
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score, fbeta_score,
    balanced_accuracy_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors
from ctgan import CTGAN
import ctgan

SEED = 42
N_SYNTHETIC = 1232                  # <-- Your requested amount
CTGAN_EPOCHS = 200                 # Increase only after evaluating results
CTGAN_BATCH_SIZE = 100            # Even and divisible by pac=10
MIN_ACCURACY = 0.60                # Required: validation accuracy strictly > 60%
MIN_RECALL = 0.80                  # Required: validation recall strictly > 80%
F_BETA = 2.0                     # F2 weights recall more heavily than precision
MIN_SPECIFICITY = 0.70            # Aspirational validation target: <=30% false positives among negatives
TARGET = "Outcome"
FEATURES = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
            "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
ZERO_IS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
INTEGER_COLUMNS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "Age"]
OUTPUT = Path("ctgan_demo_day_outputs")
OUTPUT.mkdir(exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CTGAN version:", ctgan.__version__)
print("PyTorch:", torch.__version__, "| GPU available:", torch.cuda.is_available())
print("Files will be saved to:", OUTPUT.resolve())

# Milestone 1 — Problem and raw dataset

The well-known Pima diabetes dataset contains 8 numerical predictors. One predictor (`Glucose`) is a two-hour plasma glucose measurement, so do **not** describe all eight features as non-invasive. The label represents recorded diabetes status rather than the future development of diabetes.

In [ ]:
DATA_FILE = Path("diabetes_dataset.csv")
if not DATA_FILE.exists():
    try:
        from google.colab import files
        print("Please upload diabetes_dataset.csv")
        uploaded = files.upload()
    except ImportError:
        pass
if not DATA_FILE.exists():
    raise FileNotFoundError("Upload diabetes_dataset.csv alongside this notebook and rerun this cell.")

raw = pd.read_csv(DATA_FILE)
missing_columns = sorted(set(FEATURES + [TARGET]) - set(raw.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
raw = raw[FEATURES + [TARGET]].copy()
for col in raw:
    raw[col] = pd.to_numeric(raw[col], errors="raise")
if not raw[TARGET].isin([0, 1]).all():
    raise ValueError("Outcome must contain only the binary values 0 and 1.")
raw[TARGET] = raw[TARGET].astype(int)
print("Original shape:", raw.shape)
print("Class counts (0=negative, 1=positive):")
display(raw[TARGET].value_counts().rename("Count").to_frame())
print("First five original rows:")
display(raw.head())
print("Dataset description:")
display(raw.describe().round(2))
assert len(raw) == 768, "This project expects the provided 768-row dataset. Check the input file."

# Milestone 2 — Data quality and cleaning

Some zero measurements are biologically implausible and are commonly used as **missing-value placeholders** in this dataset. We replace zero with `NaN` only for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. We **do not** alter zeros in `Pregnancies` or `Outcome`.

Crucially, **we do not fill missing values using full-dataset medians**. Medians will be learned later from training data alone, preventing leakage.

In [ ]:
print("Missing cells before cleaning:")
display(raw.isna().sum().to_frame("NaN_count"))
print("Zero values in each column before cleaning:")
display((raw == 0).sum().to_frame("Zero_count"))
print("Exact duplicate rows before cleaning:", raw.duplicated().sum())

clean = raw.drop_duplicates().copy()
for c in ZERO_IS_MISSING:
    clean.loc[clean[c] == 0, c] = np.nan

quality_table = pd.DataFrame({
    "Raw_zero_count": (raw == 0).sum(),
    "Missing_after_zero_to_NaN": clean.isna().sum(),
    "Missing_percent": (100 * clean.isna().mean()).round(1)
})
display(quality_table)
print("Rows after removing exact duplicates:", len(clean))
print("No imputation has happened yet.")
raw.describe().to_csv(OUTPUT / "01_raw_descriptive_statistics.csv")
clean.describe().to_csv(OUTPUT / "02_pre_imputation_statistics.csv")
quality_table.to_csv(OUTPUT / "03_data_quality_audit.csv")
clean.to_csv(OUTPUT / "04_cleaned_before_train_imputation.csv", index=False)

# Milestone 3A — Split **real records before generating synthetic data**

Use stratified **60% training / 20% validation / 20% testing** on *real observations only*. CTGAN must see only the original **training** records. Both validation and test sets stay completely real. We will only use validation data to pick model/threshold; the test set is evaluated once at the end.

In [ ]:
# Store original row identity so splits can be audited later.
real = clean.copy()
real.insert(0, "OriginalRowID", real.index.astype(int))
train_and_val, real_test = train_test_split(
    real, test_size=0.20, stratify=real[TARGET], random_state=SEED
)
real_train, real_val = train_test_split(
    train_and_val, test_size=0.25, stratify=train_and_val[TARGET], random_state=SEED
)
for label, frame in [("train", real_train), ("validation", real_val), ("test", real_test)]:
    print(f"{label:10} {len(frame):4} real rows | positive cases: {int(frame[TARGET].sum())}")

assert len(real_train) + len(real_val) + len(real_test) == len(real)
assert not (set(real_train.OriginalRowID) & set(real_val.OriginalRowID))
assert not (set(real_train.OriginalRowID) & set(real_test.OriginalRowID))
assert not (set(real_val.OriginalRowID) & set(real_test.OriginalRowID))

X_train, y_train = real_train[FEATURES].copy(), real_train[TARGET].astype(int).copy()
X_val, y_val = real_val[FEATURES].copy(), real_val[TARGET].astype(int).copy()
X_test, y_test = real_test[FEATURES].copy(), real_test[TARGET].astype(int).copy()
for name, split in [("train", real_train), ("validation", real_val), ("test", real_test)]:
    split.to_csv(OUTPUT / f"05_real_{name}_split.csv", index=False)

# Milestone 3B — EDA using real **training** records

We inspect the data distribution, correlations, clinical-value outliers, and class proportions. Correlation is **not causation**. The EDA is limited to the training split to avoid informing modeling decisions from the held-out test cases.

In [ ]:
plt.figure(figsize=(5.5, 4))
plt.bar(["0", "1"], [int((y_train == 0).sum()), int((y_train == 1).sum())], color=["#4477aa", "#ee8866"])
plt.title("Real training data — class distribution")
plt.xlabel("Outcome"); plt.ylabel("Number of records")
plt.tight_layout(); plt.savefig(OUTPUT / "06_real_training_class_balance.png", dpi=150); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feature in zip(axes.flat, FEATURES):
    sns.histplot(X_train[feature].dropna(), bins=25, kde=True, ax=ax, color="#4477aa")
    ax.set_title(feature)
plt.suptitle("Feature distributions — real training observations", fontsize=14)
plt.tight_layout(); plt.savefig(OUTPUT / "07_feature_distributions_real.png", dpi=150); plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(pd.concat([X_train, y_train], axis=1).corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Training-data correlations (not causal relationships)")
plt.tight_layout(); plt.savefig(OUTPUT / "08_correlation_heatmap_real.png", dpi=150); plt.show()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pd.concat([X_train, y_train], axis=1),
                x="Glucose", y="BMI", hue=TARGET, alpha=0.8, palette="Set2")
plt.title("Glucose and BMI by Outcome — real training records")
plt.tight_layout(); plt.savefig(OUTPUT / "09_glucose_BMI_scatter.png", dpi=150); plt.show()

print("Training-only feature correlations with Outcome:")
display(pd.concat([X_train, y_train], axis=1).corr()[TARGET].drop(TARGET)
        .sort_values(ascending=False).round(3).to_frame("Correlation"))

# Milestone 3C — Fit **CTGAN** on the real training portion

**Exactly 1,232 synthetic records are created by CTGAN.** It cannot learn directly from missing continuous values, so we fill those values with medians learned from the **real training split only**.

**FP reduction change:** Rather than forcing a 50/50 positive-negative distribution, generate synthetic records in approximately the **same class proportions as the original real training data** (about 35% positive). Artificially increasing the diabetes-positive rate can encourage too many positive predictions. We will still favor recall using a validation-selected threshold. This change is a **hypothesis to test**, not a guaranteed improvement.

CTGAN conditions on the true `Outcome` column to generate each label. Invalid values and exact duplicates are rejected. The final collection remains **768 real + 1,232 synthetic = 2,000 rows**.


In [ ]:
# Median imputer is fitted strictly on REAL training records.
generator_imputer = SimpleImputer(strategy="median")
generator_real_X = pd.DataFrame(
    generator_imputer.fit_transform(X_train), columns=FEATURES, index=X_train.index
)
ctgan_fit_table = generator_real_X.copy()
ctgan_fit_table[TARGET] = y_train.to_numpy().astype(int)
assert not ctgan_fit_table.isna().any().any()

print("Training CTGAN on", len(ctgan_fit_table), "real training rows only...")
print("CTGAN is NOT trained on validation or test observations.")
ctgan_model = CTGAN(
    epochs=CTGAN_EPOCHS,
    batch_size=CTGAN_BATCH_SIZE,
    pac=10,
    embedding_dim=64,
    generator_dim=(128, 128),
    discriminator_dim=(128, 128),
    verbose=True,
    enable_gpu=torch.cuda.is_available()
)
ctgan_model.fit(ctgan_fit_table, discrete_columns=[TARGET])
print("CTGAN training completed.")

## Generate **exactly 1,232** CTGAN records and enforce reasonable data types

CTGAN may produce decimal-valued `Age`, `Pregnancies`, etc. We round those integer-valued fields and limit values to the *real training* ranges. Exact duplicates of observed training rows and duplicated synthetic rows are rejected. This clipping is a pragmatic quality control, not proof of clinical realism.

In [ ]:
def postprocess_ctgan_samples(sample: pd.DataFrame, real_training_table: pd.DataFrame) -> pd.DataFrame:
    """Convert GAN output to the same schema; use ONLY real training ranges."""
    df = sample[FEATURES + [TARGET]].copy()
    for c in FEATURES + [TARGET]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    # Drop unrecognized classes; do NOT assign them the requested class.
    df = df[df[TARGET].isin([0, 1])].copy()
    df[TARGET] = df[TARGET].astype(int)
    for c in FEATURES:
        lower, upper = real_training_table[c].min(), real_training_table[c].max()
        df[c] = df[c].clip(lower=lower, upper=upper)
    for c in INTEGER_COLUMNS:
        df[c] = df[c].round().astype(int)
    return df.reset_index(drop=True)

# Generate the 1,232 synthetic records with the REAL TRAINING class ratio.
# Reason: balancing synthetic diabetes labels to 50/50 can shift the model's
# learned positive rate upward and may cause more false-positive predictions.
# Maintain the training population's approximately 35% positive prevalence.
real_positive_rate = float(y_train.mean())
n_synth_positive = int(round(N_SYNTHETIC * real_positive_rate))
n_synth_negative = N_SYNTHETIC - n_synth_positive
assert n_synth_positive + n_synth_negative == N_SYNTHETIC
print(f"Real training positive rate: {real_positive_rate:.1%}")
print("CTGAN synthetic targets: positive =", n_synth_positive,
      "| negative =", n_synth_negative)

# An all-real-training key list for exact-copy prevention.
real_keys = set(map(tuple, ctgan_fit_table[FEATURES + [TARGET]].itertuples(index=False, name=None)))

def generate_ctgan_for_label(label: int, required: int, seen_rows: set) -> pd.DataFrame:
    """Condition CTGAN on the class until we collect exactly 'required' unique rows."""
    accepted = []
    for attempt in range(1, 21):
        remaining = required - len(accepted)
        if remaining <= 0:
            break
        # Generate a little extra because some rows may fail validation.
        request_count = max(100, int(remaining * 1.25) + 25)
        batch = ctgan_model.sample(
            request_count, condition_column=TARGET, condition_value=int(label)
        )
        batch = postprocess_ctgan_samples(batch, ctgan_fit_table)
        batch = batch.loc[batch[TARGET] == label].drop_duplicates()
        for row in batch[FEATURES + [TARGET]].itertuples(index=False, name=None):
            if row not in seen_rows:
                seen_rows.add(row)
                accepted.append(row)
                if len(accepted) == required:
                    break
    if len(accepted) != required:
        raise RuntimeError(
            f"CTGAN only generated {len(accepted)}/{required} unique samples for Outcome={label}. "
            "Try increasing CTGAN_EPOCHS or reducing strict uniqueness requirements."
        )
    return pd.DataFrame(accepted, columns=FEATURES + [TARGET])

all_seen = set(real_keys)
synth_positive = generate_ctgan_for_label(1, n_synth_positive, all_seen)
synth_negative = generate_ctgan_for_label(0, n_synth_negative, all_seen)
synthetic = pd.concat([synth_negative, synth_positive], ignore_index=True)
synthetic = synthetic.sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(synthetic) == N_SYNTHETIC
assert synthetic[TARGET].sum() == n_synth_positive
assert not synthetic.duplicated().any()
assert not synthetic.isna().any().any()
assert synthetic[TARGET].isin([0, 1]).all()

synthetic.to_csv(OUTPUT / "10_CTGAN_exactly_1232_synthetic_only.csv", index=False)
print(f"SUCCESS: {len(synthetic)} new records generated by trained CTGAN.")
display(synthetic.head())
display(synthetic[TARGET].value_counts().rename("Count").to_frame())

## Prepare the **2,000-row collection** and the evaluation-stage training subset

The exported collection contains **all 768 real observations plus 1,232 CTGAN-generated records**. During **evaluation**, only the original *real training split* plus synthetic records is used for fitting; real validation/test are reserved for unbiased assessment. **After testing**, the final selected model will be trained on **all 2,000** records.

`DataSplit` labels record which real rows were held out *during evaluation*; they **do not** restrict the final full-data training.


In [ ]:
# Label ORIGINAL rows with their existing (untouched) split identity.
real_labeled = pd.concat([
    real_train.assign(DataSource="REAL", DataSplit="train"),
    real_val.assign(DataSource="REAL", DataSplit="validation"),
    real_test.assign(DataSource="REAL", DataSplit="test")
], ignore_index=True)
synthetic_labeled = synthetic.copy()
synthetic_labeled.insert(0, "OriginalRowID", np.nan)
synthetic_labeled["DataSource"] = "CTGAN_SYNTHETIC"
synthetic_labeled["DataSplit"] = "train"

all_2000 = pd.concat([real_labeled, synthetic_labeled], ignore_index=True)
model_train_1692 = pd.concat([
    real_train[FEATURES + [TARGET]].reset_index(drop=True), synthetic
], ignore_index=True)
model_train_only_X = model_train_1692[FEATURES]
model_train_only_y = model_train_1692[TARGET].astype(int)
assert len(all_2000) == len(raw) + N_SYNTHETIC == 2000
assert len(model_train_1692) == len(real_train) + N_SYNTHETIC
assert (all_2000.DataSource == "CTGAN_SYNTHETIC").sum() == 1232
assert ((all_2000.DataSource == "CTGAN_SYNTHETIC") & (all_2000.DataSplit != "train")).sum() == 0

all_2000.to_csv(OUTPUT / "11_FULL_2000_REAL_PLUS_CTGAN.csv", index=False)
model_train_1692.to_csv(OUTPUT / "12_MODEL_TRAINING_REAL_PLUS_CTGAN.csv", index=False)

counts = pd.crosstab(all_2000.DataSplit, all_2000.DataSource)
print("Final dataset shape:", all_2000.shape, "| Rows:", len(all_2000))
display(counts)
print("Final training class distribution:")
display(model_train_only_y.value_counts().rename("Count").to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("Number of records")
ax.set_title("2,000 total records — synthetic data is training-only")
plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(OUTPUT / "13_full_dataset_split_chart.png", dpi=150); plt.show()

# Milestone 3D — Synthetic data quality checks

We compare each feature distribution for synthetic versus real **training** observations (not validation/test), report Kolmogorov–Smirnov statistics and nearest-neighbor distances, and check for exact copies. A low KS statistic suggests distributions are similar, but **no statistic proves synthetic records are medically correct**. Even if exact duplicates are absent, near-copies can exist, so do not describe these records as privacy-safe by default.

### Three correlation heatmaps (adapted from your uploaded notebook)
We compare **the eight original medical feature correlations**, not the target label.
1. **Real:** correlations among actual *training* observations after training-only imputation.
2. **Synthetic:** correlations among the 1,232 generated CTGAN observations.
3. **Absolute difference:** `abs(real_corr - synthetic_corr)` for each pair of features.

The charts appear **directly in Colab** and are saved as PNG files. Smaller differences suggest closer *pairwise correlation* agreement but do **not** prove medical correctness. The synthetic dataset has a different label distribution because we requested class balancing; this can also change aggregate correlations.


In [ ]:
ks_rows = []
for c in FEATURES:
    real_c = ctgan_fit_table[c].dropna()
    synth_c = synthetic[c].dropna()
    ks_rows.append({
        "Feature": c,
        "RealTrain_Mean": real_c.mean(),
        "CTGAN_Mean": synth_c.mean(),
        "RealTrain_Std": real_c.std(),
        "CTGAN_Std": synth_c.std(),
        "KS_Statistic": ks_2samp(real_c, synth_c).statistic
    })
quality = pd.DataFrame(ks_rows).round(4)
quality.to_csv(OUTPUT / "14_CTGAN_quality_summary.csv", index=False)
display(quality)

# Distance measured in a space standardized using only training data.
qc_scaler = StandardScaler().fit(generator_real_X)
real_scaled = qc_scaler.transform(generator_real_X)
synth_scaled = qc_scaler.transform(synthetic[FEATURES])
nearest_dist, _ = NearestNeighbors(n_neighbors=1).fit(real_scaled).kneighbors(synth_scaled)
print("Minimum standardized Euclidean distance to a real training record:",
      round(float(nearest_dist.min()), 4))
print("Median standardized Euclidean distance:",
      round(float(np.median(nearest_dist)), 4))
print("Exact copies of real training rows after postprocessing: 0 (rejected during sampling).")

fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for ax, c in zip(axs.flat, FEATURES):
    ax.hist(ctgan_fit_table[c], bins=20, density=True, alpha=0.5, label="Real train")
    ax.hist(synthetic[c], bins=20, density=True, alpha=0.5, label="CTGAN")
    ax.set_title(c)
axs[0,0].legend()
plt.suptitle("Real training versus CTGAN distributions — distributional comparison")
plt.tight_layout(); plt.savefig(OUTPUT / "15_CTGAN_real_vs_synthetic_distributions.png", dpi=150); plt.show()

In [ ]:
# -------------------------------------------------------------
# THREE CORRELATION HEATMAPS (based on your High-Fidelity notebook)
# NOTE: Use REAL TRAINING records only; never look at val/test here.
# -------------------------------------------------------------
correlation_real = ctgan_fit_table[FEATURES].corr(method="pearson")
correlation_synthetic = synthetic[FEATURES].corr(method="pearson")
correlation_delta = (correlation_real - correlation_synthetic).abs()

# Save the underlying numerical matrices for your presentation/report.
correlation_real.to_csv(OUTPUT / "15A_real_training_correlation_matrix.csv")
correlation_synthetic.to_csv(OUTPUT / "15B_CTGAN_correlation_matrix.csv")
correlation_delta.to_csv(OUTPUT / "15C_absolute_correlation_difference.csv")

# Separate charts can be downloaded for PowerPoint, plus one 3-panel overview.
heatmap_specs = [
    (correlation_real, "1. Real Training Feature Correlations",
     "15A_real_training_correlations.png", "coolwarm", -1.0, 1.0),
    (correlation_synthetic, "2. CTGAN Synthetic Feature Correlations",
     "15B_CTGAN_synthetic_correlations.png", "coolwarm", -1.0, 1.0),
    (correlation_delta, "3. Absolute Correlation Difference |Real - CTGAN|",
     "15C_absolute_correlation_difference.png", "Reds", 0.0,
     max(0.4, float(correlation_delta.to_numpy().max())))
]

for matrix, title, filename, palette, lo, hi in heatmap_specs:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap=palette,
                vmin=lo, vmax=hi, linewidths=0.5,
                square=True, ax=ax, cbar=True)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", labelrotation=45)
    ax.tick_params(axis="y", labelrotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT / filename, dpi=180, bbox_inches="tight")
    plt.show()

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, (matrix, title, filename, palette, lo, hi) in zip(axes, heatmap_specs):
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap=palette,
                vmin=lo, vmax=hi, linewidths=0.4,
                square=True, ax=ax, cbar=False, annot_kws={"size": 8})
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", labelrotation=60)
    ax.tick_params(axis="y", labelrotation=0)
fig.suptitle("CTGAN Correlation Audit: Real vs Synthetic vs Difference", y=1.02, fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT / "15D_three_correlation_heatmaps_side_by_side.png",
            dpi=180, bbox_inches="tight")
plt.show()

# Compare only unique feature pairs; diagonal entries always equal 0.
upper_triangle = np.triu_indices(len(FEATURES), k=1)
unique_correlation_differences = correlation_delta.to_numpy()[upper_triangle]
avg_corr_difference = float(np.mean(unique_correlation_differences))
max_corr_difference = float(np.max(unique_correlation_differences))
print(f"Mean absolute pairwise correlation difference: {avg_corr_difference:.4f}")
print(f"Maximum absolute pairwise correlation difference: {max_corr_difference:.4f}")
print("Reminder: correlation similarity does NOT establish clinical validity or privacy safety.")
pd.DataFrame([{
    "Mean_Absolute_Pairwise_Correlation_Difference": avg_corr_difference,
    "Max_Absolute_Pairwise_Correlation_Difference": max_corr_difference,
    "Real_Training_Rows": len(ctgan_fit_table),
    "CTGAN_Rows": len(synthetic)
}]).to_csv(OUTPUT / "15E_CTGAN_correlation_audit_summary.csv", index=False)


# Milestone 4A — Feature engineering and **the four models**

The model types and hyperparameters follow the uploaded `Diabetes_Final_HighFidelity_Pipeline.ipynb`: Logistic Regression, KNN, Decision Tree, and Random Forest. Feature engineering, imputation, and scaling happen inside each scikit-learn pipeline so the validation/test data do not influence fitting. Training on the full 2,000-row collection occurs only **after** independent model evaluation.


In [ ]:
class ClinicalFeatures(BaseEstimator, TransformerMixin):
    """Create interpretable feature interactions without learning from held-out data."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        data = pd.DataFrame(X, columns=FEATURES).copy()
        data["Glucose_x_BMI"] = data["Glucose"] * data["BMI"] / 1000.0
        data["Age_x_BMI"] = data["Age"] * data["BMI"] / 1000.0
        return data

# The exact FOUR algorithms and hyperparameters in your uploaded notebook.
# All receive StandardScaler, matching that notebook's base comparison.
MODEL_SPECS = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=9, weights="distance"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=4, random_state=SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_depth=6, min_samples_leaf=3, random_state=SEED
    )
}

def make_pipeline(model_name):
    """Protect scaling/imputation from training/validation/test leakage.

    Use a fresh clone of the selected model, so each training regime has
    independent model and scaler parameters. Validation/test are transformed
    with the scaler fitted ONLY on the corresponding training data.
    """
    return Pipeline([
        ("features", ClinicalFeatures()),
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", clone(MODEL_SPECS[model_name]))
    ])

print("Four models copied from your High-Fidelity notebook:")
for name, spec in MODEL_SPECS.items():
    print(f"  • {name}: {spec}")


# Milestone 4B — Cross-validation of **real-only baselines**

Four-fold stratified cross-validation supplies a reproducible baseline for models trained on real observations. We **do not** naively cross-validate the already-augmented dataset: CTGAN learned from every real training row, which would contaminate CV folds. The CTGAN comparison instead uses a separate untouched **real validation** set in the next stage.

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
cv_rows = []
for model_name in MODEL_SPECS:
    crossval = cross_validate(
        make_pipeline(model_name), X_train, y_train,
        scoring={"Accuracy": "accuracy", "Recall": "recall", "ROC_AUC": "roc_auc"},
        cv=cv, n_jobs=1, error_score="raise"
    )
    row = {"Model": model_name}
    for name in ["Accuracy", "Recall", "ROC_AUC"]:
        row[f"CV_{name}_mean"] = float(np.mean(crossval[f"test_{name}"]))
        row[f"CV_{name}_std"] = float(np.std(crossval[f"test_{name}"]))
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("CV_ROC_AUC_mean", ascending=False)
display(cv_results.round(3))
cv_results.to_csv(OUTPUT / "16_real_only_4fold_CV.csv", index=False)

# Milestone 4C — Model comparison: **recall first, reduce false positives next**

The project still compares the same **four models**—Logistic Regression, KNN, Decision Tree, and Random Forest—trained with real data alone and with real + CTGAN data. Only the CTGAN-augmented variants are eligible for the final full-2,000-record fit.

For each model, vary the decision threshold on the **real validation split, never on the test split**:
1. **Mandatory goals:** recall strictly above **80%**, accuracy strictly above **60%**.
2. Among thresholds meeting both, **prefer fewer false positives** (higher specificity); use F2 and recall as tie-breakers.
3. The extra goal of **specificity at least 70%** is desirable, but **never silently relax recall** in an attempt to achieve it.
4. If no model meets the two mandatory goals on validation, disclose the failure and choose a documented closest alternative.

The selected threshold is frozen before evaluating the untouched real test records. Lower FP on validation **does not guarantee** lower FP on test.


In [ ]:
# Threshold optimization uses the REAL validation set only, not the final test.
# First enforce sensitivity > 80% and accuracy > 60%; THEN minimize FP.
# F2 is reported and is a tie-breaker, not the main optimization target.
THRESHOLD_GRID = np.round(np.arange(0.10, 0.801, 0.025), 3)

def calculate_metrics(y_true, probability, threshold):
    preds = (np.asarray(probability) >= float(threshold)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    a = float(accuracy_score(y_true, preds))
    r = float(recall_score(y_true, preds, zero_division=0))
    p = float(precision_score(y_true, preds, zero_division=0))
    return {
        "Accuracy": a, "Recall": r, "Precision": p,
        "F1": float(f1_score(y_true, preds, zero_division=0)),
        "F2": float(fbeta_score(y_true, preds, beta=F_BETA, zero_division=0)),
        "Specificity": float(tn / (tn+fp)) if (tn+fp) else np.nan,
        "BalancedAccuracy": float(balanced_accuracy_score(y_true, preds)),
        "ROC_AUC": float(roc_auc_score(y_true, probability)),
        "TrueNegative": int(tn), "FalsePositive": int(fp),
        "FalseNegative": int(fn), "TruePositive": int(tp),
        "PredictedPositivePercent": float(100 * preds.mean()),
    }

def select_threshold(y_true, probability):
    """Among valid thresholds, minimize false positives without sacrificing recall floor."""
    probs = np.asarray(probability, dtype=float)
    thresholds = THRESHOLD_GRID  # pre-declared grid avoids tuning to every individual validation score
    records = []
    auc = float(roc_auc_score(y_true, probs))
    for t in thresholds:
        row = calculate_metrics(y_true, probs, float(t))
        row["Threshold"] = float(t)
        row["ROC_AUC"] = auc
        row["MeetsBothGoals"] = bool(row["Accuracy"] > MIN_ACCURACY and row["Recall"] > MIN_RECALL)
        row["MeetsSpecificityGuardrail"] = bool(row["Specificity"] >= MIN_SPECIFICITY)
        row["MeetsAllGuardrails"] = bool(row["MeetsBothGoals"] and row["MeetsSpecificityGuardrail"])
        row["WeightedShortfall"] = (
            3.0 * max(0.0, MIN_RECALL - row["Recall"])
            + max(0.0, MIN_ACCURACY - row["Accuracy"])
        )
        records.append(row)
    table = pd.DataFrame(records)
    # Avoid optimizing specificity at the expense of the recall requirement.
    valid_all = table.loc[table["MeetsAllGuardrails"]]
    valid_user = table.loc[table["MeetsBothGoals"]]
    if not valid_all.empty:
        pool = valid_all
        # SPECIFICITY is equivalent to minimizing FP here: same real negatives.
        best = pool.sort_values(
            ["Specificity", "Accuracy", "F2", "Recall", "Threshold"],
            ascending=[False, False, False, False, False]
        ).iloc[0].to_dict()
    elif not valid_user.empty:
        # If 70% specificity cannot be met, still require BOTH user targets.
        pool = valid_user
        best = pool.sort_values(
            ["Specificity", "Accuracy", "F2", "Recall", "Threshold"],
            ascending=[False, False, False, False, False]
        ).iloc[0].to_dict()
    else:
        # No feasible threshold: document that the requirements are unmet.
        best = table.sort_values(
            ["WeightedShortfall", "Specificity", "F2", "Accuracy", "Threshold"],
            ascending=[True, False, False, False, False]
        ).iloc[0].to_dict()
    return best, table

validation_rows = []
threshold_search_rows = []
trained_pipelines = {}

for regime, xx, yy in [
    ("REAL_ONLY", X_train, y_train),
    ("REAL_PLUS_CTGAN", model_train_only_X, model_train_only_y)
]:
    print("\nTraining regime:", regime, "| training rows:", len(xx))
    for name in MODEL_SPECS:
        pipe = make_pipeline(name)
        pipe.fit(xx, yy)
        probs = pipe.predict_proba(X_val)[:, 1]
        best, candidates = select_threshold(y_val, probs)
        best.update({"Model": name, "TrainingRegime": regime, "TrainRows": len(xx)})
        candidates["Model"], candidates["TrainingRegime"] = name, regime
        threshold_search_rows.append(candidates)
        validation_rows.append(best)
        trained_pipelines[(regime, name)] = pipe
        print(f"  {name:25s} Accuracy={best['Accuracy']:.3f} "
              f"Recall={best['Recall']:.3f} F2={best['F2']:.3f} "
              f"Specificity={best['Specificity']:.3f} threshold={best['Threshold']:.3f} "
              f"user_goals={best['MeetsBothGoals']} specificity_guardrail={best['MeetsSpecificityGuardrail']}")

validation_results = pd.DataFrame(validation_rows)
validation_results["EligibleForFinalSelection"] = (
    validation_results["TrainingRegime"] == "REAL_PLUS_CTGAN"
)
# For display, show ALL eight results. Only CTGAN candidates are eligible.
validation_results = validation_results.sort_values(
    ["TrainingRegime", "MeetsBothGoals", "F2", "Recall", "Accuracy"],
    ascending=[True, False, False, False, False]
).reset_index(drop=True)
threshold_search_all = pd.concat(threshold_search_rows, ignore_index=True)
threshold_search_all.to_csv(OUTPUT / "17B_all_validation_threshold_candidates.csv", index=False)
validation_results.to_csv(OUTPUT / "17_model_comparison_REAL_vs_CTGAN_validation.csv", index=False)
print("\nFOUR MODEL VALIDATION COMPARISON (real validation observations)")
print(f"Threshold requirements: Recall >{MIN_RECALL:.0%}; Accuracy >{MIN_ACCURACY:.0%}.")
print("Eligible final candidates: REAL_PLUS_CTGAN only.")
display(validation_results[[
    "TrainingRegime", "Model", "TrainRows", "Accuracy", "Recall", "F2",
    "Precision", "Specificity", "FalsePositive", "FalseNegative",
    "ROC_AUC", "Threshold", "MeetsBothGoals", "MeetsSpecificityGuardrail", "EligibleForFinalSelection"
]].round(3))

plt.figure(figsize=(9, 6))
sns.scatterplot(data=validation_results, x="Accuracy", y="Recall",
                hue="TrainingRegime", style="Model", s=150)
plt.axvline(MIN_ACCURACY, linestyle="--", alpha=.6, label="Accuracy target 60%")
plt.axhline(MIN_RECALL, linestyle="--", alpha=.6, label="Recall target 80%")
plt.xlim(.4, 1.0); plt.ylim(.4, 1.0)
plt.title("Validation model comparison: accuracy vs recall")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig(OUTPUT / "18_accuracy_vs_recall_validation.png", dpi=150); plt.show()

comparison_long = validation_results.melt(
    id_vars=["Model", "TrainingRegime"], value_vars=["Accuracy", "Recall", "F2"],
    var_name="Metric", value_name="Score"
)
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
for ax, metric in zip(axes, ["Accuracy", "Recall", "F2"]):
    subset = comparison_long[comparison_long["Metric"] == metric]
    sns.barplot(data=subset, x="Model", y="Score", hue="TrainingRegime", ax=ax,
                errorbar=None)
    ax.set_title(f"{metric} — real validation set")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", labelrotation=25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT / "18B_four_models_accuracy_recall_F2.png", dpi=160, bbox_inches="tight")
plt.show()


## Choose a **CTGAN-augmented** final-model family

We select only from the four CTGAN-augmented model types, because the final refit must train on all 2,000 real-plus-synthetic rows. The ranking requires **recall >80% and accuracy >60%**; among qualifying options we prioritize **higher specificity / fewer false positives** and use accuracy and F2 as tie-breakers. We aim for 70% specificity, but that is **not guaranteed**.

A poor trade-off cannot be hidden: if no candidate passes both mandatory validation targets, the output explicitly flags the unmet goals. Even if validation passes, the independent real test can score differently.


In [ ]:
ctgan_candidates = validation_results.loc[
    validation_results["TrainingRegime"] == "REAL_PLUS_CTGAN"
].copy()
assert len(ctgan_candidates) == 4, "One CTGAN-augmented candidate per model is required."
if ctgan_candidates["MeetsAllGuardrails"].any():
    ctgan_candidates = ctgan_candidates.sort_values(
        ["MeetsAllGuardrails", "Specificity", "Accuracy", "F2", "Recall", "ROC_AUC"],
        ascending=[False, False, False, False, False, False]
    )
elif ctgan_candidates["MeetsBothGoals"].any():
    print("70% specificity goal not reached, but the recall/accuracy requirements still take priority.")
    ctgan_candidates = ctgan_candidates.sort_values(
        ["MeetsBothGoals", "Specificity", "Accuracy", "F2", "Recall", "ROC_AUC"],
        ascending=[False, False, False, False, False, False]
    )
else:
    ctgan_candidates = ctgan_candidates.sort_values(
        ["WeightedShortfall", "F2", "Recall", "Accuracy", "ROC_AUC"],
        ascending=[True, False, False, False, False]
    )
chosen = ctgan_candidates.iloc[0].to_dict()
selected_model = trained_pipelines[("REAL_PLUS_CTGAN", chosen["Model"])]
selected_threshold = float(chosen["Threshold"])
assert chosen["TrainingRegime"] == "REAL_PLUS_CTGAN"
print("SELECTED MODEL:", chosen["Model"])
print("EVALUATION TRAINING ROWS:", len(model_train_1692))
print(f"VALIDATION-SELECTED THRESHOLD: {selected_threshold:.3f}")
for k in ["Accuracy", "Recall", "Precision", "F2", "Specificity", "FalsePositive", "FalseNegative"]:
    print(f"{k}: {chosen[k]:.3f}")
print("Meets both strict validation goals?", bool(chosen["MeetsBothGoals"]))
print(f"Aspirational specificity (>= {MIN_SPECIFICITY:.0%}) met?", bool(chosen["MeetsSpecificityGuardrail"]))
if not chosen["MeetsBothGoals"]:
    print("NOTICE: None of the four CTGAN-augmented models exceeded BOTH thresholds on validation.")
print("The final model will be REFIT on ALL 2,000 rows AFTER held-out evaluation.")

# Plot how thresholds affect sensitivity, specificity, accuracy and F2.
selected_curve = threshold_search_all.loc[
    (threshold_search_all["TrainingRegime"] == "REAL_PLUS_CTGAN")
    & (threshold_search_all["Model"] == chosen["Model"])
].sort_values("Threshold")
fig, ax = plt.subplots(figsize=(10, 5))
for measure in ["Recall", "Accuracy", "F2", "Specificity"]:
    ax.plot(selected_curve["Threshold"], selected_curve[measure], label=measure)
ax.axhline(MIN_RECALL, linestyle="--", alpha=.5, label="Recall > 80% target")
ax.axhline(MIN_ACCURACY, linestyle="--", alpha=.5, label="Accuracy > 60% target")
ax.axhline(MIN_SPECIFICITY, linestyle="--", alpha=.4, label="Specificity 60% guardrail")
ax.axvline(selected_threshold, linestyle=":", color="black",
           label=f"Validation-selected threshold = {selected_threshold:.3f}")
ax.set(xlabel="Decision threshold", ylabel="Score", ylim=(0, 1),
       title="FP-minimizing threshold choice (subject to recall/accuracy) — REAL validation")
ax.legend(loc="best", fontsize=9)
plt.tight_layout(); plt.savefig(OUTPUT / "18C_validation_F2_threshold_tradeoff.png", dpi=160); plt.show()

# Practical Demo Day chart: false positives vs false negatives on VALIDATION.
# Never use test labels to choose a new threshold.
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(selected_curve["Threshold"], selected_curve["FalsePositive"],
        label="False positives (FP)", marker=".", markersize=3)
ax.plot(selected_curve["Threshold"], selected_curve["FalseNegative"],
        label="False negatives (FN)", marker=".", markersize=3)
ax.axvline(selected_threshold, linestyle="--", color="black",
           label=f"Chosen threshold = {selected_threshold:.3f}")
ax.set(xlabel="Threshold", ylabel="Count on REAL validation data",
       title="Validation FP vs FN — choose threshold before opening the test set")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "18D_validation_FP_FN_threshold_tradeoff.png", dpi=160)
plt.show()


# Milestone 4D — Final **honest real-only holdout evaluation**

The test data contains only original *real* observations. We evaluate the selected CTGAN-augmented model trained on the **real training split + synthetic records**; the test records have never contributed to that model's training or CTGAN training.

The displayed test metrics belong to this **pre-refit evaluation model**. We additionally compare its fixed validation-selected threshold with an untuned 0.50 threshold for transparency. After this evaluation, the final model will be retrained on **all 2,000** records; its independent test metrics cannot be taken from records it was trained on.

**Remember:** any score belongs to this independent pre-refit model, not to the later full-2,000-row fitted model. A single test set cannot prove the new method improves; run a fresh comparison before making a claim.


In [ ]:
# Evaluate once on strictly held-out REAL patient records.
final_probs = selected_model.predict_proba(X_test)[:, 1]
final_metrics = calculate_metrics(y_test, final_probs, selected_threshold)
comparison_at_05 = calculate_metrics(y_test, final_probs, 0.50)
print("\nREAL HOLDOUT TEST — PRE-REFIT CTGAN MODEL")
print("Chosen:", chosen["Model"], "| threshold from REAL validation:", selected_threshold)
report = pd.DataFrame([
    {"Rule": f"Validation-selected threshold {selected_threshold:.3f}", **final_metrics},
    {"Rule": "Untuned reference threshold 0.50", **comparison_at_05}
]).set_index("Rule")
display(report[[
    "Accuracy", "Recall", "Precision", "F2", "F1", "Specificity",
    "ROC_AUC", "TrueNegative", "FalsePositive", "FalseNegative", "TruePositive",
    "PredictedPositivePercent"
]].round(3))
print("\nConfusion matrix: TN={TrueNegative}, FP={FalsePositive}, FN={FalseNegative}, TP={TruePositive}".format(**final_metrics))
print("Targets achieved on real holdout?",
      bool(final_metrics["Recall"] > MIN_RECALL and final_metrics["Accuracy"] > MIN_ACCURACY))
if final_metrics["Recall"] == 1.0:
    print("CAUTION: Recall=1.00 means zero missed positives in THIS SMALL HOLDOUT only; not guaranteed for future patients.")
positive_count = int((y_test == 1).sum())
ci = binomtest(final_metrics["TruePositive"], positive_count).proportion_ci(confidence_level=.95, method="exact")
print(f"95% exact binomial CI for recall: [{ci.low:.3f}, {ci.high:.3f}] ({positive_count} real positive test cases)")
print("\nClassification report (fixed threshold):")
print(classification_report(y_test, (final_probs >= selected_threshold).astype(int), zero_division=0))

pd.DataFrame([{
    "Model": chosen["Model"], "TrainingRowsDuringEvaluation": len(model_train_1692),
    "ThresholdFromRealValidation": selected_threshold,
    "ActualFinalModelRefitRows": 2000,
    "ThisTestIsBeforeFullDataRefit": True,
    "Recall_95CI_Low": ci.low, "Recall_95CI_High": ci.high,
    **final_metrics
}]).to_csv(OUTPUT / "19_EVALUATED_MODEL_REAL_TEST_METRICS.csv", index=False)
pd.DataFrame({
    "OriginalRowID": real_test.OriginalRowID.to_numpy(),
    "ActualOutcome": y_test.to_numpy(),
    "PredictedProbability": final_probs,
    "PredictedOutcome": (final_probs >= selected_threshold).astype(int)
}).to_csv(OUTPUT / "20_EVALUATION_REAL_TEST_PREDICTIONS.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, (final_probs >= selected_threshold).astype(int),
    labels=[0, 1], display_labels=["Negative", "Positive"], cmap="Blues", ax=ax
)
ax.set_title("Evaluation model — independent REAL test confusion matrix")
plt.tight_layout();plt.savefig(OUTPUT / "21_real_holdout_confusion_matrix.png", dpi=150);plt.show()
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, final_probs, ax=axs[0])
axs[0].set_title("ROC — real holdout test")
PrecisionRecallDisplay.from_predictions(y_test, final_probs, ax=axs[1])
axs[1].set_title("Precision–recall — real holdout test")
plt.tight_layout();plt.savefig(OUTPUT / "22_ROC_PR_curves_real_holdout.png", dpi=150);plt.show()


# Feature importance and saving the **evaluated pre-refit model**

Permutation feature importance is measured on the real validation set. We save the **evaluation-stage** model separately because it is the only model whose real test performance is independent. The final model fitted on 2,000 records is created and saved in the **last code cell**.


In [ ]:
perm = permutation_importance(
    selected_model, X_val, y_val,
    scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=1
)
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_AUC_drop": perm.importances_mean,
    "Std_AUC_drop": perm.importances_std
}).sort_values("Mean_AUC_drop", ascending=False)
importance.to_csv(OUTPUT / "23_feature_permutation_importance.csv", index=False)
display(importance.round(4))
plt.figure(figsize=(8, 4.5))
sns.barplot(data=importance, y="Feature", x="Mean_AUC_drop", color="#4477aa")
plt.title("Feature permutation importance — REAL validation data")
plt.tight_layout(); plt.savefig(OUTPUT / "24_permutation_importance.png", dpi=150);plt.show()

# Keep the model whose independent real test performance was assessed.
with open(OUTPUT / "25_EVALUATED_PRE_REFIT_model.pkl", "wb") as handle:
    cloudpickle.dump({
        "model": selected_model,
        "threshold": selected_threshold,
        "features": FEATURES,
        "model_name": chosen["Model"],
        "training_rows": len(model_train_1692),
        "evaluation_uses_independent_real_test": True
    }, handle)
print("Saved evaluation model separately; full 2,000-row fit comes at the end.")


# Milestone 3 final check — properly transformed evaluation subsets

These matrices demonstrate that imputation/scaling were fitted **on evaluation training records only**, with real validation/test transformed afterward. These are **evaluation-stage** model-ready datasets, *not* a claim of independently assessing the final 2,000-row refit. A separate 2,000-row model-ready dataset is saved in the last cell.


In [ ]:
# Evaluation-stage feature matrices (no fitting on validation or test).
selected_training_X = model_train_only_X
selected_training_y = model_train_only_y
transformer = selected_model.named_steps["features"]
imputer = selected_model.named_steps["impute"]
scaler = selected_model.named_steps["scale"]
feature_names = list(transformer.transform(X_train).columns)
for split_name, X_part, y_part in [
    ("evaluation_train_real_plus_CTGAN", selected_training_X, selected_training_y),
    ("evaluation_validation_REAL", X_val, y_val),
    ("evaluation_test_REAL", X_test, y_test)
]:
    engineered = transformer.transform(X_part)
    scaled = scaler.transform(imputer.transform(engineered))
    prepared = pd.DataFrame(scaled, columns=feature_names)
    prepared[TARGET] = np.asarray(y_part, dtype=int)
    assert not prepared.isna().any().any()
    prepared.to_csv(OUTPUT / f"26_model_ready_{split_name}.csv", index=False)
    print(split_name, "—", prepared.shape, "fully numeric")


# Final step — **REFIT ON ALL 2,000 RECORDS** (no automatic ZIP download)

The model family and threshold were selected using real validation data; held-out real test results were recorded **before** this step. Now we construct the actual **final submission model**:

- **768 real records** + **1,232 CTGAN records** = **2,000 training records**.
- Refit the selected classifier, feature engineering, imputation, and scaler on all 2,000 records using a fresh pipeline.
- Save the final model as `29_FINAL_MODEL_TRAINED_ON_ALL_2000.pkl` and save the exact 2,000-row model-ready features.
- **No ZIP and no automatic Colab download.** Outputs remain in `ctgan_demo_day_outputs` and can be opened from Colab's file panel.

**Important for Demo Day:** The test results shown earlier belong to the pre-refit evaluation model, not to this final model that has subsequently seen all 768 real records. Its future performance needs independent external data to be assessed fairly. Generating synthetic rows cannot guarantee any desired metric.


In [ ]:
# This is the requested final model trained with ALL 2,000 records.
# It occurs AFTER the real holdout evaluation, so test scores remain honest.
assert len(all_2000) == 2000
assert (all_2000["DataSource"] == "REAL").sum() == 768
assert (all_2000["DataSource"] == "CTGAN_SYNTHETIC").sum() == 1232
assert all_2000[TARGET].isin([0, 1]).all()

X_all_2000 = all_2000[FEATURES].copy()
y_all_2000 = all_2000[TARGET].astype(int).copy()
assert len(X_all_2000) == len(y_all_2000) == 2000
assert len(model_train_1692) < len(all_2000)

full_data_model = make_pipeline(chosen["Model"])  # NEW independent pipeline
full_data_model.fit(X_all_2000, y_all_2000)
assert full_data_model is not selected_model
assert len(full_data_model.named_steps["features"].transform(X_all_2000).columns) == len(FEATURES) + 2
print("\n✅ FINAL MODEL FIT COMPLETED")
print("Model:", chosen["Model"])
print("Final training records:", len(X_all_2000))
print("  REAL:", int((all_2000.DataSource == "REAL").sum()))
print("  CTGAN:", int((all_2000.DataSource == "CTGAN_SYNTHETIC").sum()))
print("Chosen threshold (from earlier real validation):", selected_threshold)
print("Note: threshold may need recalibration when independent new real data become available.")

# Save final processed model-ready dataset (full 2000) for transparency.
feat = full_data_model.named_steps["features"].transform(X_all_2000)
imp = full_data_model.named_steps["impute"]
scl = full_data_model.named_steps["scale"]
scaled = scl.transform(imp.transform(feat))
model_ready_all = pd.DataFrame(scaled, columns=feat.columns)
model_ready_all[TARGET] = y_all_2000.to_numpy()
assert len(model_ready_all) == 2000
assert np.isfinite(model_ready_all.to_numpy()).all()
model_ready_all.to_csv(OUTPUT / "28_FINAL_MODEL_READY_ALL_2000.csv", index=False)

# Also preserve original real vs CTGAN identity in the final full-data table.
all_2000 = all_2000.copy()
all_2000["UsedToFitFinalModel"] = True
all_2000.to_csv(OUTPUT / "11_FULL_2000_REAL_PLUS_CTGAN.csv", index=False)

final_payload = {
    "model": full_data_model,
    "threshold": selected_threshold,
    "features": FEATURES,
    "model_name": chosen["Model"],
    "training_rows": 2000,
    "real_training_rows": 768,
    "ctgan_training_rows": 1232,
    "independent_evaluation_metrics_apply_to": "PRE_REFIT_EVALUATION_MODEL_ONLY"
}
final_path = OUTPUT / "29_FINAL_MODEL_TRAINED_ON_ALL_2000.pkl"
with open(final_path, "wb") as handle:
    cloudpickle.dump(final_payload, handle)
with open(final_path, "rb") as handle:
    loaded_final = cloudpickle.load(handle)
assert loaded_final["training_rows"] == 2000

# Showcase an example using THE FINAL 2,000-record model (not a diagnosis).
example_input = pd.DataFrame([{
    "Pregnancies": 2, "Glucose": 130, "BloodPressure": 76,
    "SkinThickness": 30, "Insulin": 100, "BMI": 31.0,
    "DiabetesPedigreeFunction": 0.5, "Age": 35
}])[FEATURES]
prob = float(loaded_final["model"].predict_proba(example_input)[0, 1])
flag = int(prob >= loaded_final["threshold"])
print(f"Final model example: probability={prob:.3f}, high-risk flag={flag} (educational only)")

summary = {
    "real_rows": 768, "CTGAN_generated_rows": 1232,
    "final_training_rows": int(len(X_all_2000)),
    "CTGAN_epochs": CTGAN_EPOCHS,
    "four_comparison_models": list(MODEL_SPECS.keys()),
    "selected_model": chosen["Model"],
    "selected_threshold_from_real_validation": float(selected_threshold),
    "validation_metrics_before_refit": {key: float(chosen[key]) for key in
        ["Accuracy", "Recall", "Precision", "F2", "Specificity", "ROC_AUC"]},
    "pre_refit_real_test_metrics": final_metrics,
    "independent_test_metrics_after_full_data_refit_available": False,
    "test_rows_were_used_for_final_model_refit_after_evaluation": True,
    "warning": "Held-out metrics apply ONLY to the pre-refit model; new real data are required to independently test the full-data model."
}
with open(OUTPUT / "30_project_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

print("\nFiles saved in Colab's folder:", OUTPUT.resolve())
print("• Full 2,000-row real+synthetic dataset:", OUTPUT / "11_FULL_2000_REAL_PLUS_CTGAN.csv")
print("• Final numeric feature table:", OUTPUT / "28_FINAL_MODEL_READY_ALL_2000.csv")
print("• Final all-2000 model:", final_path)
print("• Honest pre-refit real test scores:", OUTPUT / "19_EVALUATED_MODEL_REAL_TEST_METRICS.csv")
print("✅ No ZIP was created and no folder download was started.")
